<a href="https://colab.research.google.com/github/huutien113/games/blob/main/mp3_to_midi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MP3 piano → MIDI sạch (2 tay)
1. Menu **Runtime → Change runtime type → T4 GPU** (không bắt buộc, nhưng nhanh hơn nhiều)
2. Bấm ▶ ở cell bên dưới, chọn file mp3 khi hộp upload hiện ra
3. File `<tên>_clean.mid` sẽ tự tải về

Muốn chỉnh tham số: sửa dòng `OVERRIDES` ở đầu cell, đặt `MP3_PATH = "tên_file.mp3"` rồi chạy lại.

In [ ]:
# ============================================================
#  1 CELL DUY NHẤT:  mp3 (piano)  ->  MIDI thô  ->  MIDI đã dọn (2 tay)
# ============================================================

MP3_PATH  = None
REUSE_RAW = True
OVERRIDES = dict()

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "piano_transcription_inference", "pretty_midi", "librosa"], check=True)

from dataclasses import dataclass
from collections import defaultdict
import numpy as np


@dataclass
class N:
    start: float
    end: float
    pitch: int
    vel: int

    @property
    def dur(self):
        return self.end - self.start


CFG = dict(
    min_vel=20,
    min_vel_low=25,
    min_dur=0.08,
    short_dur=0.20,
    weak_ratio=0.60,
    ctx_win=0.5,
    low_pitch=36,
    high_pitch=90,
    ghost_high_pitch=76,
    ghost_high_vel_ratio=1.25,
    outlier_win=1.0,
    outlier_jump=10,
    group_win=0.05,
    nb_win=0.07,
    whole_tone_ratio=0.65,
    max_poly=8,
    max_poly_lh=3,
    max_poly_rh=4,
    hand_span=14,
    release_range=12,
    max_dur_low=1.6,
    max_dur_high=1.2,
    max_sustained_lh=2,
    max_sustained_rh=2,
)


def group_by_onset(notes, win):
    notes = sorted(notes, key=lambda n: (n.start, n.pitch))
    groups, cur = [], []
    for n in notes:
        if cur and n.start - cur[0].start > win:
            groups.append(cur)
            cur = []
        cur.append(n)
    if cur:
        groups.append(cur)
    return groups


def remove_short(notes, c):
    if not notes:
        return notes
    notes = sorted(notes, key=lambda n: n.start)
    starts = np.array([n.start for n in notes])
    vels = np.array([n.vel for n in notes], dtype=float)
    out = []
    for n in notes:
        v_floor = c["min_vel_low"] if n.pitch < 55 else c["min_vel"]
        if n.vel < v_floor or n.dur < c["min_dur"]:
            continue
        if n.dur < c["short_dur"]:
            lo = np.searchsorted(starts, n.start - c["ctx_win"], "left")
            hi = np.searchsorted(starts, n.start + c["ctx_win"], "right")
            local = float(np.median(vels[lo:hi]))
            extreme = n.pitch < c["low_pitch"] or n.pitch >= c["high_pitch"]
            weak = n.vel < c["weak_ratio"] * local
            if extreme or weak:
                continue
        out.append(n)
    return out


def remove_harmonic_ghosts(notes, c):
    if not notes:
        return notes

    stage1 = []
    for g in group_by_onset(notes, c["group_win"]):
        by_p = {n.pitch: n for n in g}
        stage1 += [n for n in g if not (
            n.dur < 0.25 and any(
                (n.pitch - k) in by_p and n.vel < 0.7 * by_p[n.pitch - k].vel
                for k in (12, 19, 24)))]

    stage1 = sorted(stage1, key=lambda n: n.start)
    starts = np.array([n.start for n in stage1])
    ref_notes = [n for n in stage1 if 52 <= n.pitch < c["ghost_high_pitch"]]
    ref_starts = np.array([n.start for n in ref_notes]) if ref_notes else np.array([])
    ref_pitches = np.array([n.pitch for n in ref_notes]) if ref_notes else np.array([])

    keep = []
    for i, n in enumerate(stage1):
        if n.pitch >= c["ghost_high_pitch"]:
            near_lo = np.searchsorted(starts, n.start - 0.25, "left")
            near_hi = np.searchsorted(starts, n.start + 0.25, "right")
            has_high_neighbor = any(
                j != i and abs(stage1[j].pitch - n.pitch) <= 6 and stage1[j].pitch >= c["ghost_high_pitch"] - 4
                for j in range(near_lo, near_hi)
            )
            if not has_high_neighbor:
                lo = np.searchsorted(starts, n.start - 0.06, "left")
                hi = np.searchsorted(starts, n.start + 0.06, "right")
                concurrent_low = [
                    stage1[j] for j in range(lo, hi)
                    if j != i and stage1[j].pitch <= n.pitch - 12
                ]
                if concurrent_low:
                    if len(ref_starts):
                        r_lo = np.searchsorted(ref_starts, n.start - c["outlier_win"], "left")
                        r_hi = np.searchsorted(ref_starts, n.start + c["outlier_win"], "right")
                        local_med = float(np.median(ref_pitches[r_lo:r_hi])) if r_hi > r_lo else 64.0
                    else:
                        local_med = 64.0

                    if (n.pitch - local_med) >= c["outlier_jump"] and any(
                        n.vel <= c["ghost_high_vel_ratio"] * m.vel for m in concurrent_low
                    ):
                        continue
        keep.append(n)
    return keep


def kill_neighbors(notes, c):
    notes = sorted(notes, key=lambda n: n.start)
    dead = set()
    for i, a in enumerate(notes):
        for b in notes[i + 1:]:
            if id(a) in dead:
                break
            if b.start - a.start > c["nb_win"]:
                break
            if id(b) in dead:
                continue
            d = abs(a.pitch - b.pitch)
            if d == 1:
                dead.add(id(a if a.vel < b.vel else b))
            elif d == 2 and c["whole_tone_ratio"] > 0:
                weak, strong = (a, b) if a.vel < b.vel else (b, a)
                if weak.vel < c["whole_tone_ratio"] * strong.vel:
                    dead.add(id(weak))
    return [n for n in notes if id(n) not in dead]


def cap_polyphony(notes, c):
    keep = []
    for g in group_by_onset(notes, c["group_win"]):
        if len(g) > c["max_poly"]:
            g = sorted(g, key=lambda n: -n.vel)[:c["max_poly"]]
        keep += g
    return keep


def fix_same_pitch(notes):
    by = defaultdict(list)
    for n in notes:
        by[n.pitch].append(n)
    kept = []
    for lst in by.values():
        lst.sort(key=lambda n: n.start)
        cur = lst[0]
        for nx in lst[1:]:
            if nx.start - cur.start < 0.08:
                cur.end = max(cur.end, nx.end)
                cur.vel = max(cur.vel, nx.vel)
                continue
            if nx.start < cur.end:
                cur.end = max(cur.start + 0.03, nx.start - 0.005)
            kept.append(cur)
            cur = nx
        kept.append(cur)
    return kept


def split_hands(notes, c):
    lh_pos, rh_pos = 48.0, 72.0
    L, R = [], []
    for g in group_by_onset(notes, c["group_win"]):
        ps = sorted(g, key=lambda n: n.pitch)
        best_cost, best_k = None, 0
        for k in range(len(ps) + 1):
            lo, hi = ps[:k], ps[k:]
            cost = 0.0
            for hand, pos, is_left in ((lo, lh_pos, True), (hi, rh_pos, False)):
                if not hand:
                    continue
                p = [n.pitch for n in hand]
                m = float(np.mean(p))
                cost += abs(m - pos)
                cost += 4 * max(0, (max(p) - min(p)) - c["hand_span"])
                cost += 0.5 * (max(0, m - 64) if is_left else max(0, 60 - m))
            if best_cost is None or cost < best_cost:
                best_cost, best_k = cost, k
        lo, hi = ps[:best_k], ps[best_k:]
        if len(lo) > c["max_poly_lh"]:
            lo = sorted(lo, key=lambda n: -n.vel)[:c["max_poly_lh"]]
        if len(hi) > c["max_poly_rh"]:
            hi = sorted(hi, key=lambda n: -n.vel)[:c["max_poly_rh"]]
        if lo:
            lh_pos = 0.6 * lh_pos + 0.4 * float(np.mean([n.pitch for n in lo]))
        if hi:
            rh_pos = 0.6 * rh_pos + 0.4 * float(np.mean([n.pitch for n in hi]))
        L += lo
        R += hi
    return L, R


def shorten(hand_notes, c, max_sustained=2):
    groups = group_by_onset(hand_notes, c["group_win"])
    rng = c["release_range"]
    for gi, g in enumerate(groups):
        for n in g:
            for gj in range(gi + 1, len(groups)):
                nxt = groups[gj]
                if nxt[0].start >= n.end:
                    break
                near = [m.start for m in nxt if rng is None or abs(m.pitch - n.pitch) <= rng]
                if near:
                    n.end = max(min(near), n.start + c["min_dur"])
                    break
            cap = c["max_dur_low"] if n.pitch < 60 else c["max_dur_high"]
            n.end = min(n.end, n.start + cap)

    for gi, g in enumerate(groups):
        t = g[0].start
        active_prev = [
            n for prev_g in groups[:gi] for n in prev_g
            if n.end > t + 0.02
        ]
        keep_old = max(0, max_sustained - len(g))
        if len(active_prev) > keep_old:
            active_prev.sort(key=lambda n: (n.start, n.vel))
            for n in active_prev[:len(active_prev) - keep_old]:
                n.end = max(t, n.start + c["min_dur"])


def process_notes(notes, c):
    notes = remove_short(notes, c)
    notes = remove_harmonic_ghosts(notes, c)
    notes = kill_neighbors(notes, c)
    notes = cap_polyphony(notes, c)
    notes = fix_same_pitch(notes)
    L, R = split_hands(notes, c)
    shorten(L, c, max_sustained=c["max_sustained_lh"])
    shorten(R, c, max_sustained=c["max_sustained_rh"])
    kept = {id(n) for n in fix_same_pitch(L + R)}
    L = [n for n in L if id(n) in kept]
    R = [n for n in R if id(n) in kept]
    for n in L + R:
        n.end = max(n.end, n.start + 0.03)
    return L, R


def clean_midi(in_path, out_path, **overrides):
    import pretty_midi
    c = {**CFG, **overrides}
    pm = pretty_midi.PrettyMIDI(in_path)
    notes = [N(n.start, n.end, n.pitch, n.velocity)
             for inst in pm.instruments if not inst.is_drum for n in inst.notes]
    print(f"Ban đầu: {len(notes)} nốt")

    L, R = process_notes(notes, c)
    print(f"Sau khi dọn: {len(L) + len(R)} nốt (tay phải {len(R)}, tay trái {len(L)})")

    tempi = pm.get_tempo_changes()[1]
    out = pretty_midi.PrettyMIDI(initial_tempo=float(tempi[0]) if len(tempi) else 120.0)
    for name, hand in (("Right Hand", R), ("Left Hand", L)):
        inst = pretty_midi.Instrument(program=0, name=name)
        for n in sorted(hand, key=lambda n: n.start):
            inst.notes.append(pretty_midi.Note(velocity=int(n.vel), pitch=int(n.pitch),
                                               start=float(n.start), end=float(n.end)))
        out.instruments.append(inst)
    out.write(out_path)
    return out_path


def run(mp3_path=None, reuse_raw=True, **overrides):
    import librosa, torch
    from google.colab import files
    from piano_transcription_inference import PianoTranscription, sample_rate

    if mp3_path is None:
        up = files.upload()
        mp3_path = list(up.keys())[0]
    stem = os.path.splitext(os.path.basename(mp3_path))[0]
    raw_path, out_path = f"{stem}_raw.mid", f"{stem}_clean.mid"

    if reuse_raw and os.path.exists(raw_path):
        print(f"Dùng lại {raw_path} (bỏ qua bước nhận diện)")
    else:
        audio, _ = librosa.load(mp3_path, sr=sample_rate, mono=True)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        PianoTranscription(device=device).transcribe(audio, raw_path)

    clean_midi(raw_path, out_path, **overrides)
    files.download(out_path)


run(MP3_PATH, REUSE_RAW, **OVERRIDES)

Saving Hướng Dẫn -Tình Khúc Vàng - Đan Trường - Piano hay nè.mp3 to Hướng Dẫn -Tình Khúc Vàng - Đan Trường - Piano hay nè.mp3
Checkpoint path: /root/piano_transcription_inference_data/note_F1=0.9677_pedal_F1=0.9186.pth
Total size: ~165 MB
Using cpu for inference.
Using CPU.
Segment 0 / 57
Segment 1 / 57
Segment 2 / 57
Segment 3 / 57
Segment 4 / 57
Segment 5 / 57
Segment 6 / 57
